###### 2/17/2026
## Initial Regression models for Climate Data


#### Variable and Parameters Names
 - Climate = the original data set (146 observations with 19 parameters)
 - months = Monthly climate data
 - J.D = Jan - Dec (Yearly temp)
 - N.D = Dec - Nov (Meteorological Year)
 - DJF = Dec, Jan, Feb
 - MAM = Mar, Apr, May
 - JJA = June, July, Aug
 - SON = Sep, Oct, Nov

In [59]:
## importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from pmdarima.utils import as_series
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import TimeSeriesSplit, train_test_split

## TimeSeriesSplit uses forward-chaining CV (aka Rolling-Origin Cross Validation)

plt.style.use('bmh')


#### Loading and processing data

In [60]:
Climate = pd.read_csv("Data/Global means.csv")
# Handling missing data:
## Cleaning Missing Values, by averaging over the data we have. 
labelsDJ = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov']
labelsDJF = ['Jan', 'Feb']

Climate.at[0,"D-N"] = Climate.loc[0,labelsDJ].mean().round(2)
Climate.at[0,"DJF"] =  Climate.loc[0,labelsDJF].mean()

Climate.head()

,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,J-D,D-N,DJF,MAM,JJA,SON
0,1880,-0.19,-0.25,-0.10,-0.17,-0.11,-0.22,-0.19,-0.11,-0.15,-0.24,-0.23,-0.18,-0.18,-0.18,-0.22,-0.13,-0.17,-0.21
1,1881,-0.20,-0.15,0.03,0.04,0.05,-0.19,0.00,-0.04,-0.16,-0.22,-0.19,-0.07,-0.09,-0.1,-0.18,0.04,-0.08,-0.19
2,1882,0.15,0.13,0.04,-0.17,-0.15,-0.23,-0.17,-0.08,-0.15,-0.24,-0.17,-0.36,-0.12,-0.09,0.07,-0.09,-0.16,-0.19
3,1883,-0.30,-0.37,-0.13,-0.18,-0.17,-0.08,-0.07,-0.14,-0.21,-0.11,-0.23,-0.11,-0.18,-0.2,-0.34,-0.16,-0.10,-0.19
4,1884,-0.13,-0.08,-0.36,-0.41,-0.34,-0.36,-0.30,-0.27,-0.27,-0.25,-0.34,-0.31,-0.29,-0.27,-0.11,-0.37,-0.31,-0.29


#### Running general regression and multiple regression. 
I don't know what to take as the multiple parameters. I suppose you could run the simple regression as a flattened data set of with J-D being the target, and maybe just year as a single parameter? Then split into seasonal parameters and monthly parameters? Combining seasonal and monthly data may cause singularity as they are directly correlated. I do not think any of these will produce a stronger model than the general time series model???

So maybe J-D = b + year(x)
         J-D = b + DJF(x) + MAM(x) + JJA(x) + SON(x)
         J-D = d + sum(months(x))?

In [61]:
### Simple regression on only J-D ~ Year
y = Climate['J-D']
X = sm.add_constant(Climate['Year'])

model = sm.OLS(y, X).fit()
model.summary()
mse = np.mean((model.predict(X)  - Climate["J-D"])**2)
print(f'Mean Squared Error for complete data: {mse:.6f}')

Mean Squared Error for complete data: 0.037361


In [69]:
### Simple regression on only J-D ~ Year with classical Cross Validation
y = Climate['J-D']
X = sm.add_constant(Climate['Year'])

test_size = .1
train_size = 1 - test_size
for i in range(1,10):
    X_train, X_test, y_train, y_test = split = train_test_split(X, y, test_size = .2)
    model = sm.OLS(y_train, X_train).fit()
    mse = np.mean((model.predict(X_test) - y_test)**2)
    mae = np.mean(abs(model.predict(X_test) - y_test))
    print(f'Classical fold {i}, with {train_size*10}% train  {test_size*10}% test')
    print(f'                   Mean Squared Error:  {mse:.5f}')
    print(f'                   Mean Absolute Error:  {mae:.5f}')
    print(f'                       R-Squared:  {model.rsquared:.4f}')
    print("----------------------------------------------------")

Classical fold 1, with 9.0% train  1.0% test
                   Mean Squared Error:  0.03084
                   Mean Absolute Error:  0.14396
                       R-Squared:  0.7596
----------------------------------------------------
Classical fold 2, with 9.0% train  1.0% test
                   Mean Squared Error:  0.03737
                   Mean Absolute Error:  0.15196
                       R-Squared:  0.7544
----------------------------------------------------
Classical fold 3, with 9.0% train  1.0% test
                   Mean Squared Error:  0.03591
                   Mean Absolute Error:  0.16819
                       R-Squared:  0.7677
----------------------------------------------------
Classical fold 4, with 9.0% train  1.0% test
                   Mean Squared Error:  0.03615
                   Mean Absolute Error:  0.17253
                       R-Squared:  0.7782
----------------------------------------------------
Classical fold 5, with 9.0% train  1.0% test
       

In [68]:
### Simple regression on only J-D ~ Year with Rolling-Origins
y = Climate['J-D']
X = sm.add_constant(Climate['Year'])

split = TimeSeriesSplit(n_splits = 10)

for i, (train_index, test_index) in enumerate(split.split(X)):
    X_train = X.loc[train_index,:]
    y_train = y[train_index]
    X_test = X.loc[test_index,:]
    y_test = y[test_index]
    model = sm.OLS(y_train, X_train).fit()
    mse = np.mean((model.predict(X_test) - y_test)**2)
    mae = np.mean(abs(model.predict(X_test) - y_test))
    print(f'Rolling fold {i}, with {len(train_index)/len(y):.2f} train   {len(test_index)/len(y):.2f} test')
    print(f'                   Mean Squared Error:  {mse:.5f}')
    print(f'                   Mean Absolute Error:  {mae:.5f}')
    print(f'                       R-Squared:  {model.rsquared:.4f}')
    print("----------------------------------------------------")

Rolling fold 0, with 0.11 train   0.09 test
                   Mean Squared Error:  0.01964
                   Mean Absolute Error:  0.11702
                       R-Squared:  0.1689
----------------------------------------------------
Rolling fold 1, with 0.20 train   0.09 test
                   Mean Squared Error:  0.01516
                   Mean Absolute Error:  0.10583
                       R-Squared:  0.1244
----------------------------------------------------
Rolling fold 2, with 0.29 train   0.09 test
                   Mean Squared Error:  0.03046
                   Mean Absolute Error:  0.15342
                       R-Squared:  0.1249
----------------------------------------------------
Rolling fold 3, with 0.38 train   0.09 test
                   Mean Squared Error:  0.09300
                   Mean Absolute Error:  0.28230
                       R-Squared:  0.0004
----------------------------------------------------
Rolling fold 4, with 0.47 train   0.09 test
            

In [20]:
### Multiple regression on months as explanatory variables:
### No CV
y = Climate['J-D']
X = Climate[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov']]

model = sm.OLS(y, X).fit()
model.summary()

mse = np.mean((model.predict(X)  - Climate["J-D"])**2)
print(f'Mean Squared Error for complete data: {mse:.6f}')
print(f'{np.sum(model.params)}')

inference_table = pd.DataFrame({
    'Coefficient': model.params.round(4),
    'p_value': model.pvalues.round(4)
})

# Stars for significance
def stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return ''

inference_table['Stars'] = [stars(p) for p in model.pvalues]
inference_table

Mean Squared Error for complete data: 0.000105
1.003549163387215


,Coefficient,p_value,Stars
Jan,0.0827,0.0,***
Feb,0.0744,0.0,***
Mar,0.0746,0.0,***
Apr,0.0785,0.0,***
May,0.0951,0.0,***
Jun,0.1159,0.0,***
Jul,0.0908,0.0,***
Aug,0.0904,0.0,***
Sep,0.0716,0.0,***
Oct,0.1055,0.0,***


In [ ]:
## Plotting the MSE

In [34]:
### implementing cross-validation with Rolling-Origins. 
y = Climate['J-D']
X = Climate[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov']]

split = TimeSeriesSplit(n_splits = 10)
#for i, (train_index, test_index) in enumerate(split.split(X)):
#    print(f"fold {i}")
#    print(f"Train: index = {train_index}")
#    print(f"Test: index = {test_index}")

for i, (train_index, test_index) in enumerate(split.split(X)):
    X_train = X.loc[train_index,:]
    y_train = y[train_index]
    X_test = X.loc[test_index,:]
    y_test = y[test_index]
    model = sm.OLS(y_train, X_train).fit()
    mse = np.mean((model.predict(X_test) - y_test)**2)
    print(f'Rolling fold {i}, with {len(train_index)/len(y):.2f}% train   {len(test_index)/len(y):.2f}% test')
    print(f'                   Mean Squared Error:  {mse:.5f}')
    print(f'                       R-Squared:  {model.rsquared:.4f}')
    print("----------------------------------------------------")


Rolling fold 0, with 0.11% train   0.09% test
                   Mean Squared Error:  0.00049
                       R-Squared:  0.9997
----------------------------------------------------
Rolling fold 1, with 0.20% train   0.09% test
                   Mean Squared Error:  0.00034
                       R-Squared:  0.9994
----------------------------------------------------
Rolling fold 2, with 0.29% train   0.09% test
                   Mean Squared Error:  0.00015
                       R-Squared:  0.9990
----------------------------------------------------
Rolling fold 3, with 0.38% train   0.09% test
                   Mean Squared Error:  0.00021
                       R-Squared:  0.9987
----------------------------------------------------
Rolling fold 4, with 0.47% train   0.09% test
                   Mean Squared Error:  0.00008
                       R-Squared:  0.9982
----------------------------------------------------
Rolling fold 5, with 0.55% train   0.09% test
         